# 03 — Churn propensity model, drivers and targeting

Two feature sets are compared:

| set | contents | status |
|---|---|---|
| `customer` | real IBM columns only | **honest benchmark** |
| `customer_network` | + synthetic telemetry aggregates | illustrative only (signal built in) |

The pipeline uses scikit-learn when installed; otherwise a NumPy Newton–Raphson
logistic regression is used so the notebook runs anywhere.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from telco_churn import config, ingest, clean, features, quality, model, warehouse

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid", palette=["#117865", "#C0392B", "#1F2A44", "#8A94A6"])

# Real file if present, otherwise the synthetic fixture
RAW = config.DEFAULT_RAW_FILE if config.DEFAULT_RAW_FILE.exists() else config.FIXTURE_RAW_FILE
print("Using:", RAW.name, "| synthetic fixture" if "fixture" in RAW.name else "| real IBM data")

In [ ]:
raw = ingest.load_raw_telco(RAW)
customers, _ = clean.clean_telco(raw)
kpis = ingest.load_network_kpis(config.DEFAULT_KPI_FILE)
feats = features.build_features(customers, kpis)
print("sklearn available:", model.SKLEARN_AVAILABLE)

## 1. Train

In [ ]:
runs = {name: model.train_churn_model(feats, name) for name in ["customer", "customer_network"]}
pd.DataFrame({k: {"engine": r.engine, "AUC": r.metrics["roc_auc_test"], "F1@0.5": r.metrics["at_0_50"]["f1"],
                  "recall@0.5": r.metrics["at_0_50"]["recall"], "best-F1 threshold": r.metrics["best_f1_threshold"]}
              for k, r in runs.items()}).T

## 2. ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.5))
for name, r in runs.items():
    ax.plot(r.roc["fpr"], r.roc["tpr"], label=f"{name} (AUC {r.metrics['roc_auc_test']:.3f})")
ax.plot([0, 1], [0, 1], "--", color="grey"); ax.legend(); ax.set_xlabel("FPR"); ax.set_ylabel("TPR")

## 3. Drivers (odds ratios) — customer feature set

In [ ]:
drivers = runs["customer"].drivers
display(drivers.head(15))
top = drivers.head(12).iloc[::-1]
plt.figure(figsize=(7, 4.5))
plt.barh(top["feature"], top["coefficient"], color=["#C0392B" if c > 0 else "#117865" for c in top["coefficient"]])
plt.axvline(0, color="#1F2A44"); plt.title("Log-odds coefficients (red = increases churn)")

## 4. Threshold choice — precision vs recall
A retention team usually prefers **recall** (catch most churners) at the cost of some false
alarms, so the best-F1 threshold (≈0.25–0.35) is more useful than the default 0.5.

In [ ]:
y = runs["customer"].probabilities[runs["customer"].test_mask]
yt = feats.loc[runs["customer"].test_mask, "churn_flag"].to_numpy(dtype=float)
rows = [model.classification_metrics(yt, y, t) for t in np.arange(0.1, 0.9, 0.05)]
pd.DataFrame(rows)[["threshold", "precision", "recall", "f1", "accuracy"]].round(3)

## 5. Gain chart — who to call first

In [ ]:
lift = runs["customer"].lift
display(lift)
plt.figure(figsize=(6, 4))
plt.plot(lift["cumulative_customers_pct"], lift["cumulative_churners_pct"], marker="o", color="#117865")
plt.plot([0, 100], [0, 100], "--", color="grey"); plt.xlabel("% customers targeted"); plt.ylabel("% churners captured")

## 6. Score every customer and hand off to Power BI / SQL

In [ ]:
scored = model.attach_scores(feats, runs["customer"])
scored.groupby("risk_band").agg(customers=("customer_id", "size"), actual_churn=("churn_flag", "mean"),
                                avg_prob=("churn_probability", "mean"), mrr=("monthly_charges", "sum")).round(3)